In [1]:
import open3d as o3d
import numpy as np
import copy
import os
import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt
import pickle

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# investigate
- threshold: 0.02, radius=0.01,max_nn=30 -> worst
- threshold: 0.02, radius=0.05,max_nn=10 -> slightly better
- threshold: 0.02, radius=0.05,max_nn=10 -> slightly better
- threshold: 0.1, radius=0.05,max_nn=10 -> the best - 

In [18]:

threshold  = 0.2 # folder 055, threshold = 1, folder 055_2, threshold = 0.5
voxel_size = 0.02
# radius     = 0.01
# max_nn     = 20

folder     = ''
trans_init = np.identity(4)

# path = 'C:\PointCloudProject\Data\External_tanks\\tank2'
# path = 'C:\PointCloudProject\Data\External_tanks\\tank4\\top_center_aligned'
path = 'D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0'

with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    for a in range(91,92):
        # target = o3d.io.read_point_cloud(os.path.join(path,f'0{a}.txt'),format='xyz')
        target = o3d.io.read_point_cloud(os.path.join(path,'pcd_down_combined.txt'),format='xyz')
        # target = o3d.io.read_point_cloud(os.path.join(path,'tank_0_088 cluster_5.txt'),format='xyz')
        # target = target.voxel_down_sample(voxel_size)
        # target.estimate_normals()
        target.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.2,max_nn=10))    
        
        # path1 = os.path.join(path, f'0{a}',folder)
        path1 = path
        # if not os.path.exists(path1):    
        #     os.makedirs(path1,exist_ok=True)        
        
        rms = []
        for bb in range(a+1,a+2):
            if bb == 97:
                b = 86
            else:
                b = bb
            b = 10          
            # source = o3d.io.read_point_cloud(os.path.join(path,f'0{b}.txt'), format='xyz')   
            source = o3d.io.read_point_cloud(os.path.join(path,'tank_0_091 cluster_11.txt'),format='xyz')  
            # source = o3d.io.read_point_cloud(os.path.join(path,'tank_0_096 cluster_8.txt'),format='xyz')         
            # run ICP
            print("Apply point-to-plane ICP")
            reg_p2l = o3d.pipelines.registration.registration_icp(
                source, target, threshold, trans_init,
                o3d.pipelines.registration.TransformationEstimationPointToPoint(),
                o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=500))

            # write result
            source_aligned = copy.deepcopy(source).transform(reg_p2l.transformation)
            A = np.asarray(source_aligned.points)
            points = np.c_[ A, np.ones( A.shape[0] )*b ]        
            write_array2txt(path1 + f'\\0{a}_aligned_0{b}.txt',points)

            # save transformation matrix
            with open(path1 + f'\\transform_{a}{b}.pkl', 'wb') as f:
                pickle.dump(reg_p2l.transformation, f)

            rms.append(reg_p2l.inlier_rmse)
            print(rms)
            write_array2txt(path1 + f'\\rms_{a}{b}.txt',rms,fmt='%.6f')    

[Open3D DEBUG] Format xyz File D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0\pcd_down_combined.txt
[Open3D DEBUG] Read geometry::PointCloud: 1595399 vertices.
[Open3D DEBUG] [RemoveNonFinitePoints] 0 nan points have been removed.
[Open3D DEBUG] Format xyz File D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0\tank_0_091 cluster_11.txt
[Open3D DEBUG] Read geometry::PointCloud: 959805 vertices.
[Open3D DEBUG] [RemoveNonFinitePoints] 0 nan points have been removed.
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.2856, RMSE 0.1268
[Open3D DEBUG] ICP Iteration #1: Fitness 0.2884, RMSE 0.1218
[Open3D DEBUG] ICP Iteration #2: Fitness 0.2935, RMSE 0.1192
[Open3D DEBUG] ICP Iteration #3: Fitness 0.2971, RMSE 0.1168
[Open3D DEBUG] ICP Iteration #4: Fitness 0.3028, RMSE 0.1142
[Open3D DEBUG] ICP Iteration #5: Fitness 0.3115, RMSE 0.1113
[Open3D DEBUG] ICP Iteration #6: Fitness 0.3216, RMSE 0.1091
[Open3D DEBUG] IC

In [19]:

# threshold  = 0.05 # folder 055, threshold = 1, folder 055_2, threshold = 0.5
# voxel_size = 0.02
# # radius     = 0.01
# # max_nn     = 20

# folder     = ''
# trans_init = np.identity(4)

# # path = 'C:\PointCloudProject\Data\External_tanks\\tank2'
# # path = 'C:\PointCloudProject\Data\External_tanks\\tank4\\top_center_aligned'
# path = 'D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0'

# with o3d.utility.VerbosityContextManager(
#         o3d.utility.VerbosityLevel.Debug) as cm:
#         # target.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.2,max_nn=10))    
        
#         # path1 = os.path.join(path, f'0{a}',folder)
#         path1 = path
#         # if not os.path.exists(path1):    
#         #     os.makedirs(path1,exist_ok=True)        
        
#         rms = []
         
#         source = o3d.io.read_point_cloud(os.path.join(path,'091_aligned_010.txt'),format='xyz')  
#         # source = o3d.io.read_point_cloud(os.path.join(path,'tank_0_096 cluster_8.txt'),format='xyz')         
#         # run ICP
#         print("Apply point-to-plane ICP")
#         reg_p2l = o3d.pipelines.registration.registration_icp(
#             source, target, threshold, trans_init,
#             o3d.pipelines.registration.TransformationEstimationPointToPoint(),
#             o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=500))

#         # write result
#         source_aligned = copy.deepcopy(source).transform(reg_p2l.transformation)
#         A = np.asarray(source_aligned.points)
#         points = np.c_[ A, np.ones( A.shape[0] )*b ]        
#         write_array2txt(path1 + 'final.txt',points)

#         # save transformation matrix
#         with open(path1 + f'\\transform_{a}{b}.pkl', 'wb') as f:
#             pickle.dump(reg_p2l.transformation, f)

#         rms.append(reg_p2l.inlier_rmse)
#         print(rms)
#         write_array2txt(path1 + f'\\rms_{a}{b}.txt',rms,fmt='%.6f')    

[Open3D DEBUG] Format xyz File D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0\091_aligned_010.txt
[Open3D DEBUG] Read geometry::PointCloud: 959805 vertices.
[Open3D DEBUG] [RemoveNonFinitePoints] 0 nan points have been removed.
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.3954, RMSE 0.0228
[Open3D DEBUG] ICP Iteration #1: Fitness 0.3980, RMSE 0.0226
[Open3D DEBUG] ICP Iteration #2: Fitness 0.4003, RMSE 0.0224
[Open3D DEBUG] ICP Iteration #3: Fitness 0.4019, RMSE 0.0221
[Open3D DEBUG] ICP Iteration #4: Fitness 0.4026, RMSE 0.0218
[Open3D DEBUG] ICP Iteration #5: Fitness 0.4029, RMSE 0.0215
[Open3D DEBUG] ICP Iteration #6: Fitness 0.4032, RMSE 0.0213
[Open3D DEBUG] ICP Iteration #7: Fitness 0.4034, RMSE 0.0211
[Open3D DEBUG] ICP Iteration #8: Fitness 0.4033, RMSE 0.0210
[Open3D DEBUG] ICP Iteration #9: Fitness 0.4032, RMSE 0.0208
[Open3D DEBUG] ICP Iteration #10: Fitness 0.4030, RMSE 0.0206
[Open3D DEBUG] ICP Iteration #11: Fitness 

In [20]:

threshold  = 0.02 # folder 055, threshold = 1, folder 055_2, threshold = 0.5
# voxel_size = 0.02
# radius     = 0.01
# max_nn     = 20

folder     = ''
trans_init = np.identity(4)

# path = 'C:\PointCloudProject\Data\External_tanks\\tank2'
# path = 'C:\PointCloudProject\Data\External_tanks\\tank4\\top_center_aligned'
path = 'D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0'

with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
        # target.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.2,max_nn=10))    
        
        # path1 = os.path.join(path, f'0{a}',folder)
        path1 = path
        # if not os.path.exists(path1):    
        #     os.makedirs(path1,exist_ok=True)        
        
        rms = []
         
        source = o3d.io.read_point_cloud(os.path.join(path,'091_aligned_010.txt'),format='xyz')  
        # source = o3d.io.read_point_cloud(os.path.join(path,'tank_0_096 cluster_8.txt'),format='xyz')         
        # run ICP
        print("Apply point-to-plane ICP")
        reg_p2l = o3d.pipelines.registration.registration_icp(
            source, target, threshold, trans_init,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=500))

        # write result
        source_aligned = copy.deepcopy(source).transform(reg_p2l.transformation)
        A = np.asarray(source_aligned.points)
        points = np.c_[ A, np.ones( A.shape[0] )*b ]        
        write_array2txt(path1 + '\\final.txt',points)

        # save transformation matrix
        with open(path1 + f'\\transform_{a}{b}.pkl', 'wb') as f:
            pickle.dump(reg_p2l.transformation, f)

        rms.append(reg_p2l.inlier_rmse)
        print(rms)
        write_array2txt(path1 + f'\\rms_{a}{b}.txt',rms,fmt='%.6f')    

[Open3D DEBUG] Format xyz File D:/REGISTRATION DATA/External/OLD Sampled External Scans/JB-448 26ft EXT/tank_0\091_aligned_010.txt
[Open3D DEBUG] Read geometry::PointCloud: 959805 vertices.
[Open3D DEBUG] [RemoveNonFinitePoints] 0 nan points have been removed.
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.2528, RMSE 0.0121
[Open3D DEBUG] ICP Iteration #1: Fitness 0.2561, RMSE 0.0114
[Open3D DEBUG] ICP Iteration #2: Fitness 0.2567, RMSE 0.0112
[Open3D DEBUG] ICP Iteration #3: Fitness 0.2571, RMSE 0.0111
[Open3D DEBUG] ICP Iteration #4: Fitness 0.2574, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #5: Fitness 0.2579, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #6: Fitness 0.2584, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #7: Fitness 0.2588, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #8: Fitness 0.2592, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #9: Fitness 0.2597, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #10: Fitness 0.2600, RMSE 0.0110
[Open3D DEBUG] ICP Iteration #11: Fitness 

In [22]:
final_points = np.r_[points[:,:3], np.asarray(target.points)]
write_array2txt(path1 + '\\compelete.txt',final_points)

In [25]:
print("Transform points and display")
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(final_points)
o3d.visualization.draw_geometries([pcd],
                                  zoom=0.3412,
                                  front=[0.4257, -0.2125, -0.8795],
                                  lookat=[2.6172, 2.0475, 1.532],
                                  up=[-0.0694, -0.9768, 0.2024])

Transform points and display


In [26]:
from utils.read_pts import read_input_data

In [32]:
tmp = read_input_data(path, 'pcd_down_combined.txt')
points[:,-1] = 91
final_points = np.r_[points, np.asarray(tmp.values)]
write_array2txt(path1 + '\\compelete.txt',final_points)

10.0